# Steel rust detection — reproducible YOLOv8 baseline
Mohammad Mango · M4U3

Run all in a fresh Colab session. No API keys, Drive mount, or manual uploads. Select a T4 GPU for speed. CPU execution is supported but slower.

Frozen Roboflow v1 export: CC BY 4.0 as declared in the export. Original source: https://universe.roboflow.com/university-of-tebessa/corrosion-eh3ms . Adapted version: https://universe.roboflow.com/mohammad-mango/structural-steel-rust-bboxes/dataset/1 .

Original 353/44/43 split is deterministically converted to 352 train / 88 validation, with no independent test split. Near-duplicate leakage has not yet been audited. Five author photographs are qualitative external examples, not a labelled benchmark.


## 1. Install the pinned training package

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.3.221'])

## 2. Settings and local Colab output folder
Outputs are saved under /content; download the final ZIP before ending the runtime.


In [ ]:
from pathlib import Path
import json, time, hashlib, zipfile, shutil, random, platform
from datetime import datetime, timezone
import torch, ultralytics, yaml, pandas as pd
from ultralytics import YOLO
from PIL import Image, ImageDraw
from IPython.display import display
from google.colab import files

FULL_TRAIN = True
MODEL = 'yolov8n.pt'
EPOCHS = 30 if FULL_TRAIN else 5
BATCH = 16
IMGSZ = 512
SEED = 42
CONF = 0.25
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
OUT = Path('/content/Steel_Rust_M4U3') / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
WORK = Path('/content') / ('rust_' + RUN_ID)
WORK.mkdir()
started = datetime.now(timezone.utc).isoformat()
t0 = time.perf_counter()
env = {'ultralytics':ultralytics.__version__, 'torch':torch.__version__, 'python':platform.python_version(),
       'hardware':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
       'model':MODEL,'epochs':EPOCHS,'batch':BATCH,'imgsz':IMGSZ,'seed':SEED,'prediction_conf':CONF,
       'mode':'full_training' if FULL_TRAIN else 'verification_only', 'started_utc':started}
(OUT/'environment.json').write_text(json.dumps(env, indent=2))
(OUT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print(env)
print('Output folder:', OUT)

## 3. Download the frozen dataset and verify SHA256
This is the exact source archive used for the recorded baseline. The next cell preserves the documented split transformation.


In [ ]:
import urllib.request
DATA_URL = "https://github.com/mnmango88/steel-rust-detection/releases/download/v1.0/steel-rust-v1-yolov8.zip.zip"
EXPECTED_SHA256 = "32220278b1b7277d1f662244014b876b7f8298925cec0569ded3decf08aff7b7"
archive = WORK/'source.zip'
try:
    urllib.request.urlretrieve(DATA_URL, archive)
    h = hashlib.sha256()
    with archive.open('rb') as f:
        for block in iter(lambda:f.read(1<<20), b''):
            h.update(block)
    source_sha = h.hexdigest()
    assert source_sha == EXPECTED_SHA256, f'Checksum mismatch: {source_sha}'
except Exception:
    archive.unlink(missing_ok=True)
    raise
(OUT/'source_sha256.txt').write_text(source_sha)
extract_root = WORK/'raw'
extract_root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        assert (extract_root/member.filename).resolve().is_relative_to(extract_root.resolve()), 'Unsafe ZIP path'
    z.extractall(extract_root)
candidates = [p for p in extract_root.rglob('data.yaml')
              if all((p.parent/s/'images').is_dir() for s in ['train','valid','test'])]
assert len(candidates)==1, f'Expected one dataset root, found {len(candidates)}'
raw = candidates[0].parent
print('Verified dataset:', source_sha)


## 4. Validate labels and create the documented 80/20 split
No labels are changed. Counts and pixel hashes are checked. Review similar viewpoints separately; this does not establish site-independent evaluation.

In [ ]:
extensions = {'.jpg','.jpeg','.png'}
original = {s:sorted(p for p in (raw/s/'images').iterdir() if p.suffix.lower() in extensions) for s in ['train','valid','test']}
assert [len(original[s]) for s in ['train','valid','test']] == [353,44,43], 'Unexpected source split; inspect before proceeding.'
assert yaml.safe_load((raw/'data.yaml').read_text())['names'] == ['rust']
move = random.Random(SEED).choice(original['train'])
assignments = [(p, s, 'train' if s=='train' and p!=move else 'valid') for s in original for p in original[s]]
prepared = WORK/'dataset_80_20'
for s in ['train','valid']:
    for kind in ['images','labels']:
        (prepared/s/kind).mkdir(parents=True)
records=[]; hashes={}; boxes=0
for p,old,new in assignments:
    label = p.parent.parent/'labels'/(p.stem+'.txt')
    assert label.exists(), f'Missing label: {p.name}'
    im=Image.open(p).convert('RGB'); im.load()
    pixel_hash=hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
    assert pixel_hash not in hashes, f'Exact duplicate: {p.name} and {hashes.get(pixel_hash)}'
    hashes[pixel_hash]=p.name
    rows=[line for line in label.read_text().splitlines() if line.strip()]
    for row in rows:
        a=list(map(float,row.split()))
        assert len(a)==5 and a[0]==0 and all(0<=v<=1 for v in a[1:]) and min(a[3:])>0, (p.name,row)
    boxes+=len(rows)
    dest_name=old+'__'+p.name
    shutil.copy2(p,prepared/new/'images'/dest_name)
    shutil.copy2(label,prepared/new/'labels'/(Path(dest_name).stem+'.txt'))
    records.append({'original_split':old,'source_filename':p.name,'split':new,'filename':dest_name,'boxes':len(rows),'pixel_sha256':pixel_hash})
manifest=pd.DataFrame(records)
manifest.to_csv(OUT/'split_manifest.csv',index=False)
assert manifest['split'].value_counts().to_dict()=={'train':352,'valid':88}
assert boxes==4107
DATA=prepared/'data.yaml'
DATA.write_text(yaml.safe_dump({'path':str(prepared),'train':'train/images','val':'valid/images','names':{0:'rust'}}))
shutil.copy2(DATA,OUT/'data_run.yaml')
(OUT/'split_policy.txt').write_text('Original 353/44/43. Seed 42 selects one train image to move to validation; original valid and test merged into validation. Final 352/88. No independent test set. No label edits. Near-duplicate/site leakage not yet excluded. Moved image: '+move.name)
display(manifest.groupby('split').agg(images=('filename','count'),boxes=('boxes','sum')))
print('Label syntax and exact-duplicate checks passed. Moved image:',move.name)

## 5. Save five annotation examples

In [ ]:
annotation_dir=OUT/'evidence'/'annotations'
annotation_dir.mkdir(parents=True)
for p in sorted((prepared/'train'/'images').glob('*'))[:5]:
    im=Image.open(p).convert('RGB'); draw=ImageDraw.Draw(im); w,h=im.size
    lp=prepared/'train'/'labels'/(p.stem+'.txt')
    for row in lp.read_text().splitlines():
        _,x,y,bw,bh=map(float,row.split())
        draw.rectangle(((x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h),outline='magenta',width=2)
    im.save(annotation_dir/(p.stem+'.jpg'))
    display(im.resize((384,384)))

## 6. Train YOLOv8n for 30 epochs
Early stopping is disabled so the full run completes all 30 epochs. Training uses Ultralytics augmentation defaults; the source export itself had no augmentation. Full settings are saved in args.yaml. Checkpoints are written to Google Drive after each epoch.

In [ ]:
model=YOLO(MODEL)
train_t0=time.perf_counter()
model.train(data=str(DATA),epochs=EPOCHS,batch=BATCH,imgsz=IMGSZ,device=DEVICE,
            seed=SEED,deterministic=True,workers=2,patience=0,plots=True,save=True,
            project=str(OUT),name='train',exist_ok=False)
train_seconds=time.perf_counter()-train_t0
TRAIN=Path(model.trainer.save_dir)
BEST=TRAIN/'weights'/'best.pt'
assert BEST.exists()
history=pd.read_csv(TRAIN/'results.csv')
assert len(history)>=EPOCHS
print('Training seconds:',round(train_seconds), '| Weights:',BEST)
display(Image.open(TRAIN/'results.png'))

## 7. Evaluate the best weights and save the metrics table
These are validation metrics, not independent test metrics. Precision and recall use the validator operating point; the sample prediction confidence below is separately fixed at 0.25.

In [ ]:
best=YOLO(str(BEST))
metrics=best.val(data=str(DATA),split='val',imgsz=IMGSZ,batch=BATCH,device=DEVICE,plots=True,
                 project=str(OUT),name='validation',exist_ok=False)
table=pd.DataFrame([{'precision':float(metrics.box.mp),'recall':float(metrics.box.mr),
                     'mAP50':float(metrics.box.map50),'mAP50_95':float(metrics.box.map)}])
table.to_csv(OUT/'metrics.csv',index=False)
display(table)
print('Metrics are in the range 0–1.')

## 8. Save ten validation predictions
The first ten filenames in sorted order are used, without selecting only successful examples. Compare them with their labels to identify actual FP and FN cases.

In [ ]:
samples=sorted((prepared/'valid'/'images').glob('*'))[:10]
best.predict(source=[str(p) for p in samples],imgsz=IMGSZ,conf=CONF,device=DEVICE,
             save=True,save_txt=True,save_conf=True,project=str(OUT/'evidence'),name='validation_predictions')
(OUT/'validation_sample_names.txt').write_text('\n'.join(p.name for p in samples))
for p in sorted((OUT/'evidence'/'validation_predictions').glob('*.jpg')):
    display(Image.open(p).resize((384,384)))

## 9. Predict on five author photographs
Photographs supplied by Mohammad Mango, held outside training. Confidence threshold 0.25. No manual input.


In [ ]:
import base64
PHOTO_BYTES = {'Image (39).jpg': '...', 'Image (41).jpg': '...', 'Image (42).jpg': '...', 'Image (45).jpg': '...', 'Image (49).jpg': '...'}
new_dir=WORK/'new_images'
new_dir.mkdir(exist_ok=True)
accepted=[]
for name, encoded in PHOTO_BYTES.items():
    p=new_dir/name
    p.write_bytes(base64.b64decode(encoded))
    im=Image.open(p).convert('RGB')
    ph=hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
    assert ph not in hashes, f'{name} is already in the dataset'
    accepted.append(str(p))
best.predict(source=accepted,imgsz=IMGSZ,conf=CONF,device=DEVICE,save=True,save_txt=True,save_conf=True,
             project=str(OUT/'evidence'),name='new_image_predictions')
shutil.copytree(new_dir,OUT/'evidence'/'new_image_inputs',dirs_exist_ok=True)
for p in sorted((OUT/'evidence'/'new_image_predictions').glob('*.jpg')):
    display(Image.open(p).resize((384,384)))


## 10. Save run evidence and download the results
The record does not mark GitHub reproduction as verified automatically. Confirm a successful fresh Colab run from GitHub before documenting that proof.


In [ ]:
proof={**env,'finished_utc':datetime.now(timezone.utc).isoformat(),
       'training_seconds':train_seconds,'session_seconds':time.perf_counter()-t0,
       'completed_epochs':len(history),'source_sha256':source_sha,
       'split':{'train':352,'valid':88},'fresh_github_reproduction_verified':False}
(OUT/'run_record.json').write_text(json.dumps(proof,indent=2))
backup=shutil.make_archive('/content/Steel_Rust_Results_'+RUN_ID,'zip',root_dir=OUT)
files.download(backup)
print('Saved results and weights to:',OUT)